# Analyze finetunned YOLO Models

Run the `run_yolo_detailed_testing_report.py` script on multiple YOLO models, collect metrics, and compare results.


In [1]:

# 1. Set Up Environment and Install Dependencies
import os
import sys
from pathlib import Path

import torch
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image as IPImage

# Configure matplotlib for notebook
%matplotlib inline
matplotlib.rcParams['figure.max_open_warning'] = 50

print(f"Python: {sys.version}")
print(f"Torch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IS_COLAB:
    # Running in Google Colab
    BASE_DIR = Path('/computer_vision_yolo')
else:
    # Running locally
    BASE_DIR = Path.cwd().parent


PROJECT_ROOT = BASE_DIR
SCRIPT_PATH = PROJECT_ROOT / "yolo_test" / "run_yolo_validation_report.py"

# Add project root to path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "yolo_test") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "yolo_test"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Script path: {SCRIPT_PATH}")


# Dataset Selection
# Option 1: Full dataset (~100k images) - for final optimization: "bdd100k_yolo"
# Option 2: Limited dataset (representative samples) - for quick tuning: "bdd100k_yolo_limited"
DATASET_NAME = 'bdd100k_yolo_tiny'  # 'bdd100k_yolo' or 'bdd100k_yolo_limited'
DATASET_SPLT = 'test'  # 'train', 'val', or 'test'

BATCH_SIZE = 16
import sys
sys.path.append("/computer_vision_yolo/yolo_test")
# 2. Import validation functions from script
from run_yolo_detailed_testing_report import run_validation_pipeline, visualize_predictions
print("✓ Successfully imported validation functions")

# 3. Method Loop over models, run validation, and collect metrics
results_summary = []
validation_results = {}

def test_model(models_configs):

    for cfg in models_configs:
        print("=" * 80)
        print(f"Running model: {cfg['name']} | dataset={DATASET_NAME} | split={DATASET_SPLT} | IoU={cfg['iou']}")
        print("=" * 80)

        try:
            result = run_validation_pipeline(
                model_name=cfg["name"],
                dataset_name=DATASET_NAME,
                split=DATASET_SPLT,
                iou_threshold=cfg["iou"],
                base_dir=PROJECT_ROOT,
                use_wandb=True,
                save_reports=True,
                batch_size=BATCH_SIZE,
            )

            validation_results[cfg["name"]] = result

            overall = result["metrics"]["overall"]
            yolo_overall = result["metrics"]["yolo_metrics"]

            results_summary.append({
                "model_name": cfg["name"],
                "dataset": DATASET_NAME,
                "split": DATASET_SPLT,
                "iou": cfg["iou"],
                "precision_confusion": overall["precision"],
                "recall_confusion": overall["recall"],
                "f1_confusion": overall["f1"],
                "precision_yolo": yolo_overall["precision"],
                "recall_yolo": yolo_overall["recall"],
                "map50": yolo_overall["map50"],
                "map50_95": yolo_overall["map50_95"],
                "params_m": result["model_info"]["params"] / 1e6,
                "size_mb": result["model_info"]["size(MB)"],
                "fps": result["metrics"]["fps"],
                "status": "ok",
                "run_dir": str(result["run_dir"]),
            })

        except Exception as e:
            print(f"⚠️ Model {cfg['name']} failed: {e}")
            import traceback
            traceback.print_exc()
            results_summary.append({
                "model_name": cfg["name"],
                "dataset": DATASET_NAME,        # ✅ Use global variable
                "split": DATASET_SPLT,          # ✅ Use global variable
                "iou": cfg["iou"],
                "status": "error",
            })

Python: 3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]
Torch version: 2.9.1
Device: cpu
Project root: /Users/mahdy/projects/computer_vision_yolo
Script path: /Users/mahdy/projects/computer_vision_yolo/yolo_test/run_yolo_validation_report.py
✓ Successfully imported validation functions


In [2]:
# 4. Select model configurations to test

MODEL_CONFIGS = [
    {"name": "yolov10n",  "iou": 0.5},

]


test_model(MODEL_CONFIGS)


Running model: yolov10n | dataset=bdd100k_yolo_tiny | split=test | IoU=0.5
✓ Device: cpu
✓ W&B logging enabled


wandb: Currently logged in as: m3mahdy (m3mahdy-king-saud-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ Weights & Biases initialized: yolov10n_bdd100k_yolo_tiny_test_20251126_201023
✓ Dataset loaded
  Total images: 429
  Images with labels: 429
  Label files: 429

✓ Performance metadata loaded: test_performance_analysis.json
  Images with attributes: 429
✓ Model loaded from /Users/mahdy/projects/computer_vision_yolo/models/yolov10n/yolov10n.pt
YOLOv10n summary: 223 layers, 2,775,520 parameters, 0 gradients, 8.7 GFLOPs

📊 Model Information:
  Model: yolov10n
  Classes in model: 80
  Task: detect
  Parameters: 2.8M
  Model Size: 5.6 MB
  FLOPs (640x640): 8.74 GFLOPs
  Model Size: 5.6 MB

Running per-image YOLO evaluation (model.predict)...


Evaluating images: 100%|██████████| 429/429 [00:41<00:00, 10.31img/s]



✓ Per-image evaluation completed: 429 images, total_time=41.61s

OFFICIAL YOLO VALIDATION RESULTS
Precision (mean): 0.0000
Recall (mean):    0.0000
mAP@0.5:          0.0000
mAP@0.5:0.95:     0.0000
Fitness:          0.0000

⚡ Performance Metrics:
  Total Time: 41.61s
  Average Inference Time: 96160.20 ms per image
  FPS (Frames Per Second): 10.40
(Green = Correct Predictions, Red = Incorrect Predictions, White = No Predictions)

GENERATING SAMPLE COMPARISONS

Generating 6 high-resolution comparison figures with attributes...


Generating comparisons: 100%|██████████| 6/6 [00:05<00:00,  1.00it/s]


✓ Generated 6 comparison images
  Saved to: /Users/mahdy/projects/computer_vision_yolo/yolo_test/analysis_runs/yolov10n_testing_20251126_201023/sample_comparisons

GENERATING COMPREHENSIVE FAILURE ANALYSIS
Analyzing relationship between attributes and prediction accuracy...
⚠️  Training metadata not found: /Users/mahdy/projects/computer_vision_yolo/bdd100k_yolo_tiny/representative_json/train_performance_analysis.json

Generating accuracy analysis charts...

ANALYSIS SUMMARY
Overall Accuracy: N/A
Total Images: 429
Expected Objects: 9615
Matched Objects: 0

Weakest Weather Conditions:
  - snowy: 0.00% (58 images)
  - partly cloudy: 0.00% (45 images)
  - overcast: 0.00% (57 images)

Weakest Scenes:
  - residential: 0.00% (89 images)
  - city street: 0.00% (117 images)
  - parking lot: 0.00% (53 images)

Weakest Times of Day:
  - dawn/dusk: 0.00% (106 images)
  - night: 0.00% (137 images)
  - daytime: 0.00% (169 images)

Accuracy by Object Size (Scale/Distance):

✓ Comprehensive failure an


✓ Weights & Biases run completed successfully

🧹 Cleaning up model from memory...
✓ Model removed from memory
